# LatentMind V6 — Colab Notebook

**Training-first pipeline**: generates planner dataset → trains MLP head → loads agent → runs tests.

Requires: T4 or L4 GPU runtime.

In [ ]:
# Core inference + RAG
!pip install -q 'transformers>=4.46.0' 'sentence-transformers>=3.0.0' 'accelerate>=0.27.0'
print("✓ transformers, sentence-transformers, accelerate")

# Graph + math
!pip install -q 'langgraph>=0.2.0' 'bitsandbytes>=0.43.0' scipy matplotlib
print("✓ langgraph, bitsandbytes, scipy, matplotlib")

# Utils
!pip install -q jinja2 pydantic pymysql mysql-connector-python
print("✓ jinja2, pydantic, mysql connectors")

# Flash Attention 2 (optional, ~2 min build; skip if too slow)
import subprocess
print("Installing flash-attn (may take 1-2 min on first run)...")
try:
    result = subprocess.run(
        ["pip", "install", "-q", "flash-attn", "--no-build-isolation"],
        capture_output=True, timeout=120)
    if result.returncode == 0:
        print("✓ flash-attn installed (2-3x inference speedup)")
    else:
        print("⊘ flash-attn install skipped (slow build, will use standard attention)")
except subprocess.TimeoutExpired:
    print("⊘ flash-attn install skipped (slow build, will use standard attention)")

print("\n✓ Setup complete! Ready to load agent.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = "https://github.com/Hamza09Hamza/Latent-Djezzy.git"
REPO_DIR = "/content/Latent-Djezzy"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
    print(f"cloned → {REPO_DIR}")
else:
    # always pull to get the latest code
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    # clear cached v6 modules so reimports get fresh code
    for mod in list(sys.modules.keys()):
        if "v6" in mod:
            del sys.modules[mod]
    print(f"pulled latest code, cleared module cache")

os.chdir(REPO_DIR)
print("repo ready:", REPO_DIR)

# Check if trained planner head exists
head_path = "models/planner_head.pt"
if os.path.isfile(head_path):
    import torch
    size_mb = os.path.getsize(head_path) / 1e6
    print(f"✓ Trained planner head found ({size_mb:.1f} MB) — will skip training")
else:
    print("! Trained planner head NOT found — will need to train (5 min on T4)")

In [ ]:
import shutil, os

# Adjust DRIVE_DB to where interndb.sqlite lives on YOUR Drive.
DRIVE_DB   = "/content/drive/MyDrive/interndb.sqlite"
LOCAL_DB   = "/content/interndb.sqlite"

if not os.path.isfile(LOCAL_DB):
    shutil.copy(DRIVE_DB, LOCAL_DB)
    print(f"copied {os.path.getsize(LOCAL_DB):,} bytes → {LOCAL_DB}")
else:
    print("SQLite already present:", LOCAL_DB)

In [ ]:
import os

os.environ["V6_USE_SQLITE"]    = "1"
os.environ["V6_SQLITE_PATH"]   = "/content/interndb.sqlite"
os.environ["V6_4BIT"]          = "0"          # full f16 is faster than 4-bit on T4
os.environ["V6_FLASH_ATTN"]    = "1"          # Flash Attention 2 — 2-3x faster
os.environ["V6_SPECULATIVE"]   = "1"          # speculative decoding (0.5B drafter, set 0 to disable)
os.environ["V6_SLM_OVERRIDE"]  = "Qwen/Qwen2.5-Coder-3B-Instruct"
os.environ["V6_PLANNER"]       = "prototype"  # overridden to mlp after training

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"GPU: {device}")
print(f"Flash Attention:      ✓ ON")
print(f"Speculative decoding: ✓ ON")
print(f"4-bit quantization:   ✗ OFF")
print(f"Model: 3B (fastest)")

In [ ]:
import os

head_path = "models/planner_head.pt"
if os.path.isfile(head_path):
    print(f"Skipping dataset generation (trained head already exists)")
else:
    # Generate synthetic labelled examples for the planner MLP head.
    # Output: v6/data/planner_train.jsonl  (~1 273 examples)
    !python3 -m v6.planner_data
    !wc -l v6/data/planner_train.jsonl

In [ ]:
import os

head_path = "models/planner_head.pt"
if os.path.isfile(head_path):
    print(f"Planner head already trained; skipping training")
else:
    # Train 1024→256→(intent, caps) head — ~80 epochs, <5 min on T4.
    # Output: models/planner_head.pt
    !python3 -m v6.train_planner

os.environ["V6_PLANNER"] = "mlp"
print("Planner mode set to MLP.")

In [ ]:
import sys, os
sys.path.insert(0, "/content/Latent-Djezzy")

from v6.graph import LatentMindV6

print("Loading LatentMind V6 agent (this downloads/loads the SLM + BGE-M3)…")
agent = LatentMindV6()
print("Agent ready.")

In [ ]:
from v6.config import V6Config

print("\n" + "="*60)
print("Configuration Verification")
print("="*60)
print(f"SLM model:            {V6Config.slm_id()}")
print(f"Flash Attention:      {'✓ Enabled' if V6Config.USE_FLASH_ATTN else '✗ Disabled'}")
print(f"Speculative decoding: {'✓ Enabled' if V6Config.USE_SPECULATIVE else '✗ Disabled'}")
print(f"4-bit quantization:   {'✓ Enabled' if V6Config.USE_4BIT else '✗ Disabled'}")
print(f"Router max tokens:    {V6Config.ROUTER_MAX_NEW_TOKENS}")
print(f"SQL max tokens:       {V6Config.SQLGEN_MAX_NEW_TOKENS}")
print(f"Planner mode:         {V6Config.PLANNER_MODE}")
print("="*60 + "\n")
print("Ready to run queries. Try: ask('Hello, what can you do?')")

In [ ]:
import time
from v6.state import initial_state

def ask(question: str, thread: str = "main") -> None:
    """Stream node-by-node progress, then print the final answer."""
    config = {"configurable": {"thread_id": thread}}
    state  = initial_state(question, thread)

    final_answer = ""
    t0 = time.time()

    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    for event in agent.graph.stream(state, config=config, stream_mode="updates"):
        for node_name, data in event.items():
            if node_name == "plan":
                ps    = data.get("plan_scores") or {}
                score = ps.get("score", 0.0)
                intent = data.get("intent", "?")
                caps   = data.get("capabilities", [])
                print(f"  [plan]   intent={intent}  conf={score:.2f}  caps={caps}")

            elif node_name == "sql_generate":
                sql = data.get("sql", "")
                if sql:
                    short = sql.replace("\n", " ")[:80]
                    print(f"  [sql]    {short}…" if len(sql) > 80 else f"  [sql]    {short}")

            elif node_name == "sql_execute":
                rows = data.get("rows") or []
                err  = data.get("sql_error", "")
                if err:
                    print(f"  [exec]   ERROR: {err[:60]}")
                else:
                    print(f"  [exec]   {len(rows)} row(s)")

            elif node_name == "answer":
                final_answer = data.get("final_answer", "")

            elif node_name == "direct_answer":
                final_answer = data.get("final_answer", "")

            elif node_name == "finalize":
                if not final_answer:
                    final_answer = data.get("final_answer", "")

            else:
                pass  # route/retrieve/orchestrate/etc. — silent

    elapsed = time.time() - t0
    print(f"\nAnswer ({elapsed:.1f}s):")
    print(final_answer or "(no answer captured)")
    print()

In [ ]:
ask("Hello, what can you do?")

In [ ]:
ask("What is ARPU?")

In [ ]:
ask("What is the total revenue for Oran in 2024?")

In [ ]:
ask("Compare churn rates between Algiers and Constantine last quarter")

In [ ]:
ask("Show me a bar chart of monthly revenue by wilaya for Q1 2024")

In [ ]:
ask("Generate an executive report for Q4 2024 performance")

In [ ]:
ask("Which wilaya had the highest churn rate?")  # follow-up — tests memory

In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated()  / 1e9
    reserv = torch.cuda.memory_reserved()   / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  allocated: {alloc:.2f} GB")
    print(f"GPU  reserved:  {reserv:.2f} GB")
    print(f"GPU  total:     {total:.2f} GB")

In [ ]:
# Direct streaming — bypasses the graph, useful for quick SLM checks.
from v6.slm import get_slm

slm = get_slm()
messages = [
    {"role": "system", "content": "You are a helpful telecom analyst."},
    {"role": "user",   "content": "List the top 3 KPIs for a telecom operator."},
]
print("Streaming response:")
for token in slm.stream_generate(messages, max_new_tokens=256):
    print(token, end="", flush=True)
print()